In [1]:
import os
import re
from pathlib import Path
import wikipediaapi
import json
from typing import List
import numpy as np
from sentence_transformers import SentenceTransformer
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility

In [2]:
USER_AGENT = os.getenv(
    "USER_AGENT",
    "TourGuideAI/1.0 (Learning project), Mozilla/5.0 (Windows NT 10.0; Win64; x64), Chrome/91.0.4472.124 Safari/537.36")

In [3]:
wiki = wikipediaapi.Wikipedia(language='de',
                             extract_format=wikipediaapi.ExtractFormat.WIKI,
                             user_agent=USER_AGENT)

In [4]:
URL_PATH_WIKI = Path('../data/wiki')

In [5]:
def load_urls(path: Path):
    urls = []
    for file in path.glob("*.json"):
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

            for entry in data:
                urls.append(entry["url"])
    return urls
urls_wiki = load_urls(URL_PATH_WIKI)
urls_wiki

['https://de.wikipedia.org/wiki/Spreewald',
 'https://de.wikipedia.org/wiki/L%C3%BCbbenau/Spreewald',
 'https://de.wikipedia.org/wiki/Brandenburg_an_der_Havel',
 'https://de.wikipedia.org/wiki/Potsdam',
 'https://de.wikipedia.org/wiki/Potsdamer_Stadtschloss',
 'https://de.wikipedia.org/wiki/Alter_Markt_(Potsdam)',
 'https://de.wikipedia.org/wiki/Gro%C3%9Fer_Spreewaldhafen_L%C3%BCbbenau',
 'https://de.wikipedia.org/wiki/Kahn',
 'https://de.wikipedia.org/wiki/Landkreis_Oberspreewald-Lausitz',
 'https://de.wikipedia.org/wiki/Raddusch',
 'https://de.wikipedia.org/wiki/Vetschau/Spreewald',
 'https://de.wikipedia.org/wiki/Sorbisches_Siedlungsgebiet',
 'https://de.wikipedia.org/wiki/Slawenburg_Raddusch',
 'https://de.wikipedia.org/wiki/Bahnhof_Raddusch',
 'https://de.wikipedia.org/wiki/L%C3%BCbben_(Spreewald)',
 'https://de.wikipedia.org/wiki/Datei:Bootshaus_Kaupen_-_Paddelboot_im_Spreewald_-_Quodda.jpg',
 '',
 '']

In [6]:
def remove_references(text: str) -> str:
    stop_sections = ['Einzelnachweise', 'Weblinks', 'Literatur', 'Quellen', 'Fußnoten']
    for section in stop_sections:
        if section in text:
            text = text.split(section)[0]
    return text.strip()

In [7]:
def clean_text(text):
    text = remove_references(text)
     # Kleinbuchstaben, Entfernen von Sonderzeichen
    text = text.lower().replace(",", " ").replace("!", " ").replace("?", " ").replace("-", " ").replace(")", " ").replace("(", " ").strip()

    return text

In [8]:
def load_wikipedia_text(title: str) -> dict:
    page = wiki.page(title)
    if not page.exists():
        return {'error': 'Page not found'}

    text = clean_text(page.text)
    return {
        'text': text,
        'title': page.title,
        'source': page.fullurl,
        'license': 'CC BY-SA 4.0'
    }

#Titel aus URL extrahieren
def extract_title_from_url(url: str) -> str:
    match = re.search(r'/wiki/([^#]+)', url)
    if match:
        return match.group(1)
    return None

wiki_data = {}
for url in urls_wiki:
    title = extract_title_from_url(url)
    wiki_data[title] = load_wikipedia_text(title)

In [9]:
def get_summary_from_wiki(text:str, n=2) -> str:
    paragraphs = text.split('\n\n')
    summary = ' '.join(paragraphs[:n])
    return summary

#print(json.dumps(wiki_data, indent=2, ensure_ascii=False))
print(f"\n {len(wiki_data)} Wikipedia-Artikel geladen.")
for i, (title, data) in enumerate(wiki_data.items(),start=1):
    if 'text' in data:
        summary = get_summary_from_wiki(data['text'])
        print(f"\n--- Artikel {i}: ---")
        print(f"\n {data['title']}\n{summary}\n")


 17 Wikipedia-Artikel geladen.

--- Artikel 1: ---

 Spreewald
der spreewald  niedersorbisch błota  „die sümpfe“  ist ein ausgedehntes niederungsgebiet und eine historische kulturlandschaft im südosten brandenburgs. hauptmerkmal ist die natürliche flusslaufverzweigung der spree  die durch angelegte kanäle deutlich erweitert wurde. als auen  und moorlandschaft besitzt sie für den naturschutz überregionale bedeutung und ist als biosphärenreservat geschützt  siehe biosphärenreservat spreewald . der spreewald als kulturlandschaft wurde entscheidend durch die sorben geprägt. das gebiet ist eines der bekanntesten und beliebtesten reiseziele im land brandenburg. insgesamt 222 8 kilometer im oberspreewald und 45 4 kilometer im unterspreewald sind als landeswasserstraße klassifiziert. geografie
gliederung
der spreewald befindet sich in den landkreisen spree neiße  dahme spreewald und oberspreewald lausitz. er wird in den südlichen und größeren oberspreewald und den nördlichen  kleineren unters

In [10]:
def chunk_text(text: str, max_words: int = 200, overlap: int = 50) -> List[str]:
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + max_words
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = end - overlap  # Überlappung für Kontext
        if start < 0:
            start = 0

    return chunks

In [11]:
#Chunking mit metadaten
wiki_chunks = []
for title, data in wiki_data.items():
    if 'text' in data:
        chunks = chunk_text(data['text'], max_words=200, overlap=50)
        for i, chunk in enumerate(chunks):
            wiki_chunks.append({
                "text": chunk,
                "title": data['title'],
                "url": data['source']
            })
print(f"{len(wiki_chunks)} Text-Chunks mit Metadaten erstellt.")

213 Text-Chunks mit Metadaten erstellt.


In [12]:
#Milvus
connections.connect("default", host="127.0.0.1", port="19530")

# Collection-Schema definieren
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=4096),
    FieldSchema(name="title", dtype=DataType.VARCHAR, max_length=1024),
    FieldSchema(name="url", dtype=DataType.VARCHAR, max_length=1024)
]
schema = CollectionSchema(fields, description="Wikipedia Chunks für semantische Suche")

# Collection erstellen
collection_name = "wiki_collection"
if utility.has_collection(collection_name):
    utility.drop_collection(collection_name)
collection = Collection(name=collection_name, schema=schema)

In [22]:
#Embeddings für Wiki-Chunks erstellen

#model1 = "sentence-transformers/all-MiniLM-L6-v2"
model2 = "paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = SentenceTransformer(model2)
wiki_embeddings = embedding_model.encode([chunk['text'] for chunk in wiki_chunks],show_progress_bar=True).astype(np.float32).tolist()
print(f"Embeddings für {len(wiki_embeddings)} Wiki-Chunks erstellt.")

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embeddings für 213 Wiki-Chunks erstellt.


In [14]:
#Daten in Milvus einfügen
title = [chunk['title'] for chunk in wiki_chunks]
url = [chunk['url'] for chunk in wiki_chunks]
collection.insert([wiki_embeddings, [chunk['text'] for chunk in wiki_chunks], title, url])

(insert count: 213, delete count: 0, upsert count: 0, timestamp: 464847763324010500, success count: 213, err count: 0

In [18]:
#Index erstellen (HNSW)
index_params = {
    "index_type": "HNSW",
    "metric_type": "COSINE",
    "params": {"M": 16, "efConstruction": 200}
}
collection.create_index(field_name="embedding", index_params=index_params)
print(f"Index für Collection '{collection_name}' erstellt.")

collection.load()


Index für Collection 'wiki_collection' erstellt.


In [19]:
#Semantische Suche in Milvus
def semantic_search_milvus(query: str, k: int = 5):
    q_emb = embedding_model.encode([query]).astype(np.float32).tolist()
    results = collection.search(
        data=q_emb,
        anns_field="embedding",
        param={"metric_type": "COSINE", "params": {"ef": 50}},
        limit=k,
        output_fields=["text", "title", "url"]
    )
    output = []  # Da wir nur eine Query haben
    for result in results:
        for hit in result:
            output.append({
                "score": hit.score,
                "text": hit.entity.get("text"),
                "title": hit.entity.get("title"),
                "url": hit.entity.get("url")
            })
    return output

In [20]:
# Test
query = "Geschichte, Spreewald"
results = semantic_search_milvus(query, k=5)

for r in results:
    print(f"\n--- Treffer: {r['title']} ---")
    print(f"Score: {r['score']:.4f}")
    print(f"URL: {r['url']}")
    print(f"Text: {r['text'][:500]}...")  # Nur die ersten 300 Zeichen anzeigen


--- Treffer: Spreewald ---
Score: 0.4282
URL: https://de.wikipedia.org/wiki/Spreewald
Text: der spreewald niedersorbisch błota „die sümpfe“ ist ein ausgedehntes niederungsgebiet und eine historische kulturlandschaft im südosten brandenburgs. hauptmerkmal ist die natürliche flusslaufverzweigung der spree die durch angelegte kanäle deutlich erweitert wurde. als auen und moorlandschaft besitzt sie für den naturschutz überregionale bedeutung und ist als biosphärenreservat geschützt siehe biosphärenreservat spreewald . der spreewald als kulturlandschaft wurde entscheidend durch die sorben g...

--- Treffer: Spreewald ---
Score: 0.4264
URL: https://de.wikipedia.org/wiki/Spreewald
Text: 1879 den spreewald und warb dafür ihn zu besuchen: „wer den spreewald bereisen will und es ist zweck dieser darstellung dazu aufzufordern …“ dann folgen seine reiseempfehlungen . als einer der begründer des organisierten tourismus im spreewald gilt der lehrer paul fahlisch der ab 1882 touristische kahnfahrten